<a href="https://colab.research.google.com/github/MahiDeshpande/when-metrics-disagree/blob/main/when_metrics_disagree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **When Metrics Disagree: A Visual Investigation of PSNR, SSIM, LPIPS, and TPIPS**

## **The main question?**
Our main question that we are trying to answer is: What does “image quality” actually mean, and when do PSNR, SSIM, LPIPS, and newer perceptual metrics (TPIPS) agree or disagree with what humans see?

This project explores three core layers:
1. **Understanding Metric Perceptions:** Analyzing how each algorithm interprets image variations.
2. **Controlled Stress-Testing:** Subjecting benchmark images to synthetic transformations (pixel shifts, Gaussian blurs, noise, JPEG compression, and brightness shifts).
3. **Counterexample Discovery:** Finding failure modes where metric score rankings directly contradict human visual intuition.
4. **Bonus:** Going a little more in-depth into different models (SSIM vs MS-SSIM). Researching differences and/or variation when using different backbones (testing out AlexNet vs VGG for LPIPS)
---

## **Progress Tracker**

| Task / Experiment | Test Description | Status |
| :--- | :--- | :---: |
| **Setup & Dependencies** | Import libraries and initialize metric models (`skimage`, `lpips`, `tpips`). | [ ] |
| **Test 1: Spatial Translation** | Evaluate metric sensitivity on 1–5 pixel shifts. | [ ] |
| **Test 2: High-Frequency Blur** | Measure metric response across increasing Gaussian blur radii. | [ ] |
| **Test 3: Noise & Artifacts** | Stress-test with Additive Noise & JPEG compression artifacts. | [ ] |
| **Test 4: Photometric Shifts** | Evaluate uniform brightness and contrast alterations. | [ ] |
| **Counterexample Analysis** | Compile side-by-side failure mode plots showing metric disagreements. | [ ] |

---

## **Extras & Additional Notes**
- **Dataset:** Small image subset provided for benchmark testing.
- **Goal:** Export key visual plots and qualitative findings into the final technical blog post and project repository.

## **Installing Required Libraries To Run Image Quality Metrics**

OpenCV, scikit-image, PyTorch, LPIPS, Matplotlib, and HuggingFace Transfomers (for TPIPS backbones)

---

In [8]:

!pip install -q --upgrade torchao
!pip install -q opencv-python-headless scikit-image torch torchvision lpips matplotlib ftfy regex tqdm transformers
!pip install -q git+https://github.com/adobe-research/TPIPS.git

# General utils, image processing & plotting
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import userdata
from PIL import Image

# Grab Hugging Face token from Colab Secrets and set environment variable
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

# Core PyTorch setup
import torch
import torchvision.transforms as T

# Metric 1 & 2: PSNR and SSIM (classical math metrics)
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

# Metric 3: LPIPS (deep feature metric)
import lpips

# Metric 4: TPIPS (text-prompted perceptual metric)
import tpips

# Setup device (using T4 GPU enabled in runtime, else fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Load LPIPS networks into memory
print("Loading LPIPS (AlexNet & VGG)...")
lpips_alex = lpips.LPIPS(net="alex").to(device).eval()
lpips_vgg = lpips.LPIPS(net="vgg").to(device).eval()

# Load TPIPS network into memory on CPU to fit within VRAM limits
print("Loading TPIPS...")
tpips_model = tpips.load_model("embedding", device="cpu")

print("\nAll models loaded. Let's get started. ")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Running on: cuda
Loading LPIPS (AlexNet & VGG)...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/vgg.pth
Loading TPIPS...
[load_model] fetching sywang/TPIPS-Embed-Qwen3VL-8B from Hugging Face ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/749 [00:00<?, ?it/s]

[embedding] LoRA on LLM (r=16, alpha=32); 38.9M trainable / 8183.7M total


[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--sywang--TPIPS-Embed-Qwen3VL-8B/snapshots/86a1493c60121e656a1c075bc7aed88f25fcd4d4' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[load_model] embedding loaded from /root/.cache/huggingface/hub/models--sywang--TPIPS-Embed-Qwen3VL-8B/snapshots/86a1493c60121e656a1c075bc7aed88f25fcd4d4 on cpu

Ready to go! All models loaded.
